## 2.1 文本预处理 - token & Tokenization

#### 1、为什么这一小节很重要

##### 1.1 这是文本进入 RNN 的第一步
我们上一小节已经知道：

- 自然语言不能直接输入 RNN
- RNN 只能处理数值
- 文本在进入模型前，必须先变成“序列”

但是这里马上会出现一个新的问题：

一句话到底应该被拆成什么样的序列？

例如这句话：

`I love deep learning`

它可以被拆成很多不同形式：

- 按字符拆：`I / l / o / v / e / ...`
- 按单词拆：`I / love / deep / learning`
- 按更小的子词拆：`learn / ing`

所以，在真正学习词表、索引化、Embedding 之前，我们必须先理解：

文本的“基本单位”到底是什么。

##### 1.2 这一小节决定了后面整个文本处理流程
因为你选择什么作为文本单位，后面很多步骤都会跟着变化：

- 词表怎么建
- 序列长度会变成多长
- 未知词怎么处理
- Embedding 的对象是什么
- RNN 每个时间步输入的到底是什么

所以这一小节虽然还没有进入模型计算，但它是整个 NLP 流程的起点。

##### 1.3 可以把它类比为 CNN 里的“像素单位”
在 CNN 中，我们先学习：

- 图像是由像素组成的
- 像素是图像最基础的数值单位

在 NLP 中，也有一个类似的问题：

- 文本是由什么单位组成的
- 这个单位是字符、单词，还是别的东西
- 自然语言的“最小建模单位”是什么

#### 2、字符、单词、token 分别是什么

##### 2.1 字符 character
字符，就是文本中最基础的字面符号。

例如英文单词：

`cat`

按字符拆分后就是：

- `c`
- `a`
- `t`

再例如一句话：

`I love AI`

按字符拆分后可以看成：

`I / 空格 / l / o / v / e / 空格 / A / I`

如果忽略空格，常见写法可能是：

`I / l / o / v / e / A / I`

对于中文来说，单个汉字也常常可以看成字符，例如：

`我喜欢学习`

按字符拆分：

`我 / 喜 / 欢 / 学 / 习`

所以字符级处理，就是把文本看成“字符序列”。

##### 2.2 单词 word
单词，就是我们平时在英文里最直观的语言单位。

例如：

`I love deep learning`

按单词拆分后得到：

`I / love / deep / learning`

这个方式最符合人的直觉，因为我们通常会把一句英文自然看成若干个单词组成。

对于英文 NLP 初学阶段，word-level 是最容易理解的一种方式。

##### 2.3 token
token 是 NLP 里最常见、也最重要的术语之一。

可以先给出一个最实用的理解：

token = 文本切分后得到的一个基本处理单位

注意，这里故意没有说 token 一定等于单词。

因为 token 可能是：

- 一个字符
- 一个单词
- 一个子词
- 一个标点
- 一个特殊符号

也就是说：

token 是“切分后的单位”的统称，而不是某一种固定形式。

所以：

- 如果你按字符切分，字符就是 token
- 如果你按单词切分，单词就是 token
- 如果你按子词切分，子词就是 token

这就是为什么 token 这个词比 word 更通用。


#### 3、什么叫文本的基本单位

##### 3.1 文本不是一个整体块，而是由多个元素组成
对于人来说，一句话看起来像一个整体。

例如：

`I love deep learning`

但从模型处理的角度看，它不能作为一个大整体直接扔进去，而是要先拆成一串更小的元素。

这些更小的元素，就是文本的基本单位。

##### 3.2 不同任务中，基本单位可以不同
文本的基本单位并不是永远固定的。

常见的几种拆法有：

- 字符 `character`
- 单词 `word`
- 子词 `subword`

所以我们后面学习时，不要把 token 和 word 完全画等号。

虽然在初学阶段，我们常常把 token 简单理解成“一个单词”，但严格来说，token 是一个更广义的概念。

##### 3.3 模型并不直接理解语言，而是理解“被切分后的序列”
这句话非常关键：

模型不是直接理解一句完整的话，而是处理这句话切分后的结果。

例如：

`I love AI`

如果按单词切分，模型看到的是：

`I → love → AI`

如果按字符切分，模型看到的是：

`I → l → o → v → e → A → I`

你会发现，同一句话，不同切分方式，模型看到的序列完全不同。

所以“如何切分文本”，本身就是自然语言处理中的核心问题之一。

#### 4、什么是分词 Tokenization

##### 4.1 分词的本质
它的本质就是：

把原始文本切分成一个个 token

例如：

`I love deep learning`

分词后可能得到：

`I / love / deep / learning`

这就是一次最简单的 tokenization。

##### 4.2 为什么必须分词
因为原始文本是一个整体字符串，而模型需要的是一个序列。

所以中间必须经过这样一步：

原始句子  
$\rightarrow$ 切成多个单位  
$\rightarrow$ 形成序列

分词就是把“整块文本”变成“可处理序列”的第一步。

没有分词，后面的很多步骤就没法做：

- 没法建立词表
- 没法编号
- 没法变成索引序列
- 没法送入 Embedding
- 没法按时间步输入 RNN

所以分词是文本预处理中的核心起点。

##### 4.3 分词后的结果就是 token 序列
例如：

`This movie is very good`

分词后：

`This / movie / is / very / good`

那么模型后续处理的，实际上就不再是原句本身，而是这个 token 序列。

你可以把它理解为：

自然语言处理的第一刀，就是把句子“切开”。


#### 5、按字符切分、按单词切分

##### 5.1 按字符切分 character-level
字符级切分的特点：

- 词表通常比较小，因为字符种类有限
- 几乎不会遇到未知词问题，因为任何单词都能拆成字符
- 但序列会变得很长，训练成本更高
- 单个字符携带的语义较弱，模型需要自己组合出词义

举个例子：

`cat` 这个词，如果按字符输入，模型要从 `c`、`a`、`t` 三个字符里自己慢慢学出 `cat` 这个词的意义。

这对模型来说更难。

##### 5.2 按单词切分 word-level
单词级切分的特点：

- 更符合人的语言直觉
- 单个 token 往往就已经带有明确语义
- 序列长度通常更短
- 更适合初学者理解 RNN 的输入流程

但它也有一个明显问题：

未知词问题 OOV，`Out Of Vocabulary`

例如词表里有：

`I / love / AI`

但是没有：

`ChatGPT`

那么这个新单词就可能无法直接处理，只能映射成一个特殊符号。

所以，word-level 容易理解，但对新词比较敏感。

##### 5.3 二者对比总结
字符级像是“拆得很细”：

优点是灵活，不怕新词。  
缺点是序列长、语义弱。

单词级像是“直接按自然语言单位拆”：

优点是语义清晰，序列更短。  
缺点是容易出现未知词。

所以这两种方式各有优缺点。


#### 6、按照子词拆分 ✅

##### 6.1 子词是什么
子词可以理解为：

介于“字符”和“完整单词”之间的一种单位

例如：

`playing`

它不一定整体作为一个 token，也不一定拆成单个字符，而可能拆成：

`play / ing`

再例如：

`unhappy`

可能拆成：

`un / happy`

这就是子词切分的基本思想。

##### 6.2 为什么会有子词这种方式
因为字符级和单词级各有缺点：

- 字符级太细，语义弱，序列长
- 单词级太粗，容易遇到未知词

子词正好试图在两者之间找一个平衡。

它的目标是：

既尽量保留词的语义结构，又能减少未知词问题。

例如：

词表里没有 `playfully`

但如果词表里有：

`play / ful / ly`

那么这个新词仍然可以被拆解和处理。

##### 6.3 为什么现代 NLP 特别喜欢 subword
因为它比单词更灵活，又比字符更有语义。

所以在现代模型中，尤其是 Transformer、BERT、GPT 这类模型里，子词切分非常常见。

不过对于我们当前学习 RNN 基础阶段来说，不需要一上来就深入子词算法细节。

你现在只需要知道：

除了字符和单词之外，还有一种很重要的切分方式叫子词，它是现代 NLP 中非常常见的 token 形式。


#### 7、英文分词和中文分词为什么不一样

##### 7.1 英文通常有天然空格
英文句子中，单词之间通常天然就有空格。

例如：

`I love deep learning`

我们很容易按空格切分为：

`I / love / deep / learning`

所以英文在最基础阶段，分词相对简单。

##### 7.2 中文没有天然空格
中文通常是连续书写的，例如：

`我喜欢深度学习`

这里没有天然空格告诉我们应该怎么切。

它可能切成：

`我 / 喜欢 / 深度学习`

也可能切成：

`我 / 喜欢 / 深度 / 学习`

这就说明中文分词本身就是一个更复杂的问题。

##### 7.3 中文分词为什么更难
因为中文句子的边界不是直接写出来的，而是要通过规则、词典或者算法判断。

例如：

`研究生命起源`

它可以理解为：

`研究 / 生命 / 起源`

但在某些场景下，也可能出现别的歧义切法。

所以中文分词不像英文那样简单按空格 `split()` 一下就行，它通常需要专门的分词工具。


#### 8、为什么说 token 是后续一切操作的基础

##### 8.1 没有 token，就没有词表
后面我们要建立 `vocabulary` 词表。

但词表里装的是什么？

本质上装的就是 token。

- 如果你按单词切分，词表装的主要就是单词
- 如果你按字符切分，词表装的主要就是字符
- 如果你按子词切分，词表装的主要就是子词

所以：

先有 token，后有词表。

##### 8.2 没有 token，就没法编号
后面模型不能直接处理字符串，所以我们要把每个 token 变成一个数字 id。

例如：

- `I → 1`
- `love → 2`
- `AI → 3`

这一步的前提，就是你已经先把句子拆成 token 了。

##### 8.3 没有 token，就没法形成时间步序列
RNN 是按时间步逐个输入的。

那么每个时间步输入什么？

答案就是：一个 token 对应的一步输入。

所以在文本任务中，你可以这样理解：

一个 token，通常就对应 RNN 的一个时间步。

例如：

`I / love / AI`

就对应 $3$ 个时间步。